# Topología Aplicada y Computacional

## Práctica 1

Antes de crear la clase `SimplicialComplex` se desarrolla la función `power_set`, que calcula el conjunto de las partes de una tupla. La implementación aprovecha la representación en base $2$ de los números para codificar cada conjunto del conjunto potencia. De esta manera, el elemento $i$-ésimo de la tupla se incluye en cada iteración si el bit $i$-ésimo del número es $1$. Recorriendo todos los números hasta $2^N - 1$, dónde $N$ es la longitud de la tupla, se obtiene el resultado.

In [4]:
def power_set(elems: tuple[int, ...]) -> list[tuple[int, ...]]:
    """Return a list of all subsets as tuples."""
    n = len(elems)

    res = []

    for i in range(1, 1 << n):
        res.append(tuple(
            elems[j] for j in range(n) if i & (1 << j)
        ))

    return res


También se crea la clase `Simplex` para abstraer muchas de las operaciones en `SimplicialComplex`. Ésta incluye numerosos métodos que realizan tareas comunes como comprobar su dimensión en `dim`, ver si es cara o disjunto a otro símplice en `is_face` y `is_disjoint` respectivamente y computar todas sus caras en `faces`. También se implementan varios *dunder methods* para poder iterar el símplice, indexarlo, hashearlo, compararlo y representarlo.

In [5]:
from __future__ import annotations


class Simplex:
    """Represents a simplex in topology, a set of vertices defining a geometric structure."""

    def __init__(self, vertices: tuple[int, ...]) -> None:
        """Initialize the Simplex object by providing the set of vertices."""
       
        # the vertices are sorted, so that the simplices (1,2,3) and (3,2,1) get the same representation
        verts = tuple(sorted(vertices))

        if len(set(verts)) != len(verts):
            raise ValueError("A simplex cannot contain repeated vertices")

        self._vertices: tuple[int, ...] = verts

    @property
    def vertices(self) -> tuple[int, ...]:
        """Return the sorted, immutable vertices of the simplex."""
        return self._vertices

    def dim(self) -> int:
        """Return the dimension of the simplex."""
        return len(self._vertices) - 1

    def is_face(self, other: Simplex) -> bool:
        """Return True if self is a face of the given simplex."""
        return set(self._vertices).issubset(other._vertices)

    def is_disjoint(self, other: Simplex) -> bool:
        """Return True if self and other share no common vertices."""
        return set(self._vertices).isdisjoint(other._vertices)

    def faces(self) -> set[Simplex]:
        """Return a set of all face simplices."""
        return {Simplex(t) for t in power_set(self._vertices)}

    def __iter__(self):
        """Return an iterator over vertices."""
        return iter(self._vertices)

    def __getitem__(self, index: int) -> int:
        """Return the vertex at the specified index."""
        return self._vertices[index]

    def __hash__(self) -> int:
        """Return the hash value for use in sets and dictionaries."""
        return hash(self._vertices)

    def __eq__(self, other: object) -> bool:
        """Return True if self and other represent the same vertices."""
        if isinstance(other, Simplex):
            return self._vertices == other._vertices

        return NotImplemented

    def __repr__(self) -> str:
        """Return a readable representation of the simplex."""
        return f"Simplex({self._vertices})"


**Ejercicio 1:** Definir una clase para almacenar complejos simpliciales.

A continuación se implementa la clase. Para crear un objeto `SimplicialComplex` basta con proveer el iterable `generator` con los símplices iniciales, sin que haga falta añadir todas sus caras (ya que esto se hace automáticamente computando su conjunto potencia). Por otro lado, la dimensión se obtiene del símplice generador de mayor dimensión.

Se ha decidido almacenar como atributo todos los símplices del complejo simplicial, en lugar de sólo los maximales, para no tener que recomputarlos cada vez que se quiera consultar el `_face_set`. De no ser así, cada llamada supondría una operación de complejidad $O(2^N)$.

In [6]:
from collections import defaultdict
from typing import Iterable


class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            self._face_set.update(simplex.faces())

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set


En efecto, aunque el complejo simplicial sólo se defina con el triángulo $\{1,2,3\}$, las caras restantes se añaden automáticamente.

In [7]:
SimplicialComplex([Simplex((1, 2, 3))]).face_set


{Simplex((1, 2)),
 Simplex((1, 2, 3)),
 Simplex((1, 3)),
 Simplex((1,)),
 Simplex((2, 3)),
 Simplex((2,)),
 Simplex((3,))}

**Ejercicio 2:** Definir una función que devuelva la dimensión del complejo simplicial.

Como ya se precomputa la dimensión en un atributo, basta con retornarla.

In [8]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            self._face_set.update(simplex.faces())

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set


De nuevo, se comprueba su funcionamiento.

In [9]:
SimplicialComplex([Simplex((1, 2, 3))]).dim


2

**Ejercicio 3:** Definir una función que devuelva el conjunto de todas las caras del complejo simplicial.

Como el constructor ya computa el conjunto de caras y lo guarda en el atributo `_face_set`, el trabajo ya está hecho.

**Ejercicio 4:** Definir una función que devuelva el conjunto de todas las caras de dimensión dada.

Basta con filtrar las caras de la propiedad `face_set` con la dimensión indicada. También se implementa la función `skeleton` que filtra las caras con dimensieon menor o igual a la dada.

In [10]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            self._face_set.update(simplex.faces())

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}

Comprobamos su funcionamiento.

In [11]:
SimplicialComplex([Simplex((1, 2, 3, 4))]).n_faces(1)


{Simplex((1, 2)),
 Simplex((1, 3)),
 Simplex((1, 4)),
 Simplex((2, 3)),
 Simplex((2, 4)),
 Simplex((3, 4))}

**Ejercicio 5:** Definir una función que calcule la estrella de un símplice.

A partir de la función `Simplex.is_face` se implementa `st` que computa la estrella del parámetro usando la definición y emite un error en caso de que el símplice provisto no sea una cara del complejo simplicial.

In [12]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            self._face_set.update(simplex.faces())

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set
        
    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def st(self, simplex: Simplex) -> set[Simplex]:
        """Return the star of the simplex."""
        res = {member for member in self.face_set if simplex.is_face(member)}

        if not res:
            raise ValueError("'simplex' is not a face")

        return res


Finalmente, se comprueba el funcionamiento del método.

In [13]:
SimplicialComplex([Simplex((1, 2, 3))]).st(Simplex((1,)))


{Simplex((1, 2)), Simplex((1, 2, 3)), Simplex((1, 3)), Simplex((1,))}

**Ejercicio 6:** Definir una función que calcule el link de un símplice.

El link de un símplice $\tau$ es el conjunto de símplices cuya intersección con la estrella cerrada de $\tau$ es vacía. De este modo para implementar el método `link` es primero necesario un método para computar la estrella cerrada de un símplice. Esto es sencillo, ya que basta con usar el constructor de `SimplicialComplex` con la estrella abierta como generador, produciendo así el mínimo complejo simplicial que contiene la estrella. De este modo, una vez implementado `closed_st` basta con usar la definición de link y la función `Simplex.is_disjoint` para implementar el método `lk`.

In [14]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            self._face_set.update(simplex.faces())

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}

    def st(self, simplex: Simplex) -> set[Simplex]:
        """Return the star of the simplex."""
        res = {member for member in self.face_set if simplex.is_face(member)}

        if not res:
            raise ValueError("'simplex' is not a face")

        return res

    def closed_st(self, simplex: Simplex) -> SimplicialComplex:
        """Return the closed star of the simplex."""
        return SimplicialComplex(self.st(simplex))

    def lk(self, simplex: Simplex) -> set[Simplex]:
        """Return the link of the simplex."""
        closed = self.closed_st(simplex)
        return {elem for elem in closed.face_set if simplex.is_disjoint(elem)}


Se comprueba que su funcionamiento es correcto.

In [15]:
SimplicialComplex([Simplex((1, 2, 3))]).lk(Simplex((1,)))


{Simplex((2, 3)), Simplex((2,)), Simplex((3,))}

**Ejercicio 7:** Definir una función que calcule la característica de 
Euler

La característica de Euler, $\chi(K)$, del complejo simplicial $K$ se calcula usando una función auxiliar `_euler_characteristic`. Para calcularla, se itera sobre cada uno de los símplices del complejo y se tiene en cuenta que cada símplice $\tau \in K$ aporta $(-1)^{dim \; \tau}$. Así, si un símplice tiene dimensión par suma $1$ a $\chi(K)$ y, si es impar, resta $1$ a $\chi(K)$. Esta función se llama en el constructor de la clase y se almacena $\chi(K)$ en la clase como atributo de nombre `euler_characteristic`.

In [16]:
    @staticmethod
    def _euler_characteristic(face_set: set[Simplex]) -> int:
        """Compute the Euler characteristic using face dimensions."""
        chi = 0

        for face in face_set:
            if face.dim() % 2 == 0:
                chi += 1
            else:
                chi -= 1

        return chi


Y el constructor quedaría:

In [17]:
    def __init__(self, generator: Iterable[Simplex]) -> None:
        # Resto del constructor...

        self.euler_characteristic: int = self._euler_characteristic(self.face_set)


La clase `SimplicialComplex` queda como:

In [18]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            self._face_set.update(simplex.faces())

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic: int = self._euler_characteristic(self.face_set)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}
        
    def st(self, simplex: Simplex) -> set[Simplex]:
        """Return the star of the simplex."""
        res = {member for member in self.face_set if simplex.is_face(member)}

        if not res:
            raise ValueError("'simplex' is not a face")

        return res

    def closed_st(self, simplex: Simplex) -> SimplicialComplex:
        """Return the closed star of the simplex."""
        return SimplicialComplex(self.st(simplex))

    def lk(self, simplex: Simplex) -> set[Simplex]:
        """Return the link of the simplex."""
        closed = self.closed_st(simplex)
        return {elem for elem in closed.face_set if simplex.is_disjoint(elem)}
        
    @staticmethod
    def _euler_characteristic(face_set: set[Simplex]) -> int:
        """Compute the Euler characteristic using face dimensions."""
        chi = 0

        for face in face_set:
            if face.dim() % 2 == 0:
                chi += 1
            else:
                chi -= 1

        return chi

Probamos con un ejemplo

In [19]:
k = SimplicialComplex([Simplex((1, 2, 3))])
k.euler_characteristic


1

**Ejercicio 8:** Definir una función que determine el número de 
componentes conexas del espacio subyacente

Se toma el grafo $G = (V, E)$, donde $V = \text{Vert}(K)$ y $E = K^{(1)} - \text{Vert}(K)$. Es decir, $G$ es el grafo formado por los vértices y los símplices de dimensión $1$ de $K$. Cada componente conexa de $|K|$ se corresponde con una componente conexa del grafo $G$. Así, para calcular el número de componentes conexas de $|K|$ se calcula el número de componentes conexas de $G$ usando el algoritmo de búsqueda en anchura, ampliamente usado en teoría de grafos.

In [20]:
    def graph(self) -> dict[int, list[int]]:
        """Return a dictionary representing the 1-skeleton as a graph."""
        graph = defaultdict(list)

        for v1, v2 in self.n_faces(1):
            graph[v1].append(v2)
            graph[v2].append(v1)

        return graph

    def connected_components(self) -> int:
        """Return the number of connected components."""
        visited = set()

        graph = self.graph()
        vertices = [v[0] for v in self.n_faces(0)]

        components = 0

        for u in vertices:
            if u in visited:
                continue

            queue = [u]
            visited.add(u)

            while queue:
                v = queue.pop()

                for neig in graph[v]:
                    if neig not in visited:
                        queue.append(neig)
                        visited.add(neig)

            components += 1

        return components

La clase queda del siguiente modo:

In [21]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    @staticmethod
    def _euler_characteristic(face_set: set[Simplex]) -> int:
        """Compute the Euler characteristic using face dimensions."""
        chi = 0

        for face in face_set:
            if face.dim() % 2 == 0:
                chi += 1
            else:
                chi -= 1

        return chi

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            faces = simplex.faces()

            self._face_set.update(faces)

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic: int = self._euler_characteristic(self.face_set)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}

    def st(self, simplex: Simplex) -> set[Simplex]:
        """Return the star of the simplex."""
        res = {member for member in self.face_set if simplex.is_face(member)}

        if not res:
            raise ValueError("'simplex' is not a face")

        return res

    def closed_st(self, simplex: Simplex) -> SimplicialComplex:
        """Return the closed star of the simplex."""
        return SimplicialComplex(self.st(simplex))

    def lk(self, simplex: Simplex) -> set[Simplex]:
        """Return the link of the simplex."""
        closed = self.closed_st(simplex)
        return {elem for elem in closed.face_set if simplex.is_disjoint(elem)}

    def graph(self) -> dict[int, list[int]]:
        """Return a dictionary representing the 1-skeleton as a graph."""
        graph = defaultdict(list)

        for v1, v2 in self.n_faces(1):
            graph[v1].append(v2)
            graph[v2].append(v1)

        return graph

    def connected_components(self) -> int:
        """Return the number of connected components."""
        visited = set()

        graph = self.graph()
        vertices = [v[0] for v in self.n_faces(0)]

        components = 0

        for u in vertices:
            if u in visited:
                continue

            queue = [u]
            visited.add(u)

            while queue:
                v = queue.pop()

                for neig in graph[v]:
                    if neig not in visited:
                        queue.append(neig)
                        visited.add(neig)

            components += 1

        return components

Se comprueba su funcionamiento:

In [22]:
sc = SimplicialComplex([Simplex((1,2,3)), Simplex((4,5))])
sc.connected_components()

2

**Ejercicio 9:** Añade una función a la clase de complejos simpliciales que te permita 
añadir símplices nuevos. 

Esta función es similar al constructor de `SimplicalComplex`. En la entrada `generator` se proporciona una lista de símplices, se calculan todas sus caras usando el conjunto potencia y se añaden al atributo `face_set`. Por último, se actualiza la dimensión y la característica de Euler.

In [23]:
    def insert(self, generator: Iterable[Simplex]) -> None:
        """Insert a generator of simplices into the complex."""
        additions = set()

        for simplex in generator:
            difference = simplex.faces() - self.face_set

            additions.update(difference)
            self._face_set.update(difference)

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic += self._euler_characteristic(additions)

De este modo, la clase `SimplicialComplex` queda así:

In [24]:
class SimplicialComplex:
    """Represents a simplicial complex: a set of simplices closed under taking faces."""

    @staticmethod
    def _euler_characteristic(face_set: set[Simplex]) -> int:
        """Compute the Euler characteristic using face dimensions."""
        chi = 0

        for face in face_set:
            if face.dim() % 2 == 0:
                chi += 1
            else:
                chi -= 1

        return chi

    def __init__(self, generator: Iterable[Simplex]) -> None:
        """Initialize the simplicial complex with a set of simplices."""
        self.dim: int = -1
        self._face_set: set[Simplex] = set()

        for simplex in generator:
            faces = simplex.faces()

            self._face_set.update(faces)

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic: int = self._euler_characteristic(self.face_set)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the set of all simplices (treat as read-only)."""
        return self._face_set

    def n_faces(self, dim: int) -> set[Simplex]:
        """Return all simplices with the specified dimension."""
        return {face for face in self.face_set if face.dim() == dim}

    def skeleton(self, dim: int) -> set[Simplex]:
        """Return all simplices with dimension less than or equal to the specified value."""
        return {face for face in self.face_set if face.dim() <= dim}

    def st(self, simplex: Simplex) -> set[Simplex]:
        """Return the star of the simplex."""
        res = {member for member in self.face_set if simplex.is_face(member)}

        if not res:
            raise ValueError("'simplex' is not a face")

        return res

    def closed_st(self, simplex: Simplex) -> SimplicialComplex:
        """Return the closed star of the simplex."""
        return SimplicialComplex(self.st(simplex))

    def lk(self, simplex: Simplex) -> set[Simplex]:
        """Return the link of the simplex."""
        closed = self.closed_st(simplex)
        return {elem for elem in closed.face_set if simplex.is_disjoint(elem)}

    def graph(self) -> dict[int, list[int]]:
        """Return a dictionary representing the 1-skeleton as a graph."""
        graph = defaultdict(list)

        for v1, v2 in self.n_faces(1):
            graph[v1].append(v2)
            graph[v2].append(v1)

        return graph

    def connected_components(self) -> int:
        """Return the number of connected components."""
        visited = set()

        graph = self.graph()
        vertices = [v[0] for v in self.n_faces(0)]

        components = 0

        for u in vertices:
            if u in visited:
                continue

            queue = [u]
            visited.add(u)

            while queue:
                v = queue.pop()

                for neig in graph[v]:
                    if neig not in visited:
                        queue.append(neig)
                        visited.add(neig)

            components += 1

        return components

    def insert(self, generator: Iterable[Simplex]) -> None:
        """Insert a generator of simplices into the complex."""
        additions = set()

        for simplex in generator:
            difference = simplex.faces() - self.face_set

            additions.update(difference)
            self._face_set.update(difference)

            if simplex.dim() > self.dim:
                self.dim = simplex.dim()

        self.euler_characteristic += self._euler_characteristic(additions)

Vemos que funciona correctamente:

In [25]:
sc = SimplicialComplex([Simplex((1,2))])
print(sc.face_set)

sc.insert([Simplex((2,3))])
print(sc.face_set)

{Simplex((1,)), Simplex((1, 2)), Simplex((2,))}
{Simplex((2, 3)), Simplex((1,)), Simplex((1, 2)), Simplex((2,)), Simplex((3,))}


**Ejercicio 10:** Añade un nuevo input que permita añadir un flotante a los símplices de 
cada inserción.

Para realizar los ejercicios siguientes se crea un *wrapper* de `SimplicialCompex`, llamado `FilteredSimplicialComplex`. Esta nueva clase contendrá, a parte del complejo simplicial (del tipo `SimplicialComplex`), dos diccionarios:
* Un diccionario de nombre `timestamps`, que relaciona un flotante con el conjunto de símplices con ese flotante asignado.
* Un diccionario de nombre `time_of`, que asigna a cada símplice su flotante asignado.
Este segundo diccionario permite consultar el flotante asignado de un símplice dado en tiempo constante, en lugar de tener que buscarlo en el diccionario de `timestamps`.

La clase `FilteredSimplicialComplex` se define inicialmente como:

In [26]:
class FilteredSimplicialComplex:
    """Represents a simplicial complex whose simplices carry filtration timestamps."""

    def _finish_insertion(self, additions: set[Simplex], time: float) -> None:
        """Add simplices to the timestamp dictionary."""
        if not additions:
            return

        self.timestamps[time].update(additions)

        for addition in additions:
            self.time_of[addition] = time

    def __init__(self, generator: Iterable[Simplex], time: float = 0.0) -> None:
        """Initialize the filtered complex with a set of simplices."""
        self.sc: SimplicialComplex = SimplicialComplex(generator)

        self.timestamps: dict[float, set[Simplex]] = defaultdict(set)
        self.time_of: dict[Simplex, float] = {}

        self._finish_insertion(self.sc.face_set, 0.0)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the face set."""
        return self.sc.face_set

A continuación se crea la función `insert` en la clase `FilteredSimplicialComplex`, para la que se utilizará la función `SimplicialComplex.insert`, definida en el ejercicio anterior. 

Primero se aíslan los símplices del generador provisto que no estuviesen ya en el símplice y se almacenan en la variable `additions`. A continuación, se verifica la propiedad que tiene que tener el argumento `time`: Para cada símplice nuevo $\tau$, se tiene que si $\sigma \le \tau$ entonces $\text{float}(\sigma) \le \text{float}(\tau)$. Esto se comprueba solo con los símplices en `additions` y usando el diccionario `time_of` con cada cara de dicho símplice. Una vez realizada dicha verificación, se insertan los nuevos símplices y se actualizan los valores de los diccionarios `timestamps` y `time_of` con el método `FilteredSimplicialComplex._finish_insertion`.

In [27]:
class FilteredSimplicialComplex:
    """Represents a simplicial complex whose simplices carry filtration timestamps."""

    def _finish_insertion(self, additions: set[Simplex], time: float) -> None:
        """Add simplices to the timestamp dictionary."""
        if not additions:
            return

        self.timestamps[time].update(additions)

        for addition in additions:
            self.time_of[addition] = time

    def __init__(self, generator: Iterable[Simplex], time: float = 0.0) -> None:
        """Initialize the filtered complex with a set of simplices."""
        self.sc: SimplicialComplex = SimplicialComplex(generator)

        self.timestamps: dict[float, set[Simplex]] = defaultdict(set)
        self.time_of: dict[Simplex, float] = {}

        self._finish_insertion(self.sc.face_set, time)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the face set."""
        return self.sc.face_set
        
    def insert(self, generator: Iterable[Simplex], time: float) -> None:
        """Insert a generator of simplices at the given timestamp."""
        generator = list(generator)

        for simplex in generator:
            if simplex in self.sc.face_set:
                raise ValueError("'generator' contains an already added simplex")

        additions: set[Simplex] = set()

        for simplex in generator:
            additions.update(simplex.faces())

        additions -= self.sc.face_set

        for addition in additions:
            for face in addition.faces():
                t = self.time_of.get(face)

                if t is not None and t > time:
                    raise ValueError("'generator' contains a face with a bad time value")

        self.sc.insert(generator)
        self._finish_insertion(additions, time)

Comprobamos que funciona correctamente y que se detectan las violaciones de la condición:

In [28]:
from IPython.display import display

fsc = FilteredSimplicialComplex([Simplex((1,2))], 0.0)
display(fsc.timestamps)

fsc.insert([Simplex((1,2,3))], 1.0)    # funciona correctamente
display(fsc.timestamps)

try:
    fsc.insert([Simplex((3,4))], 0.5)  # produce un error
except ValueError as e:
    display(e)

defaultdict(set, {0.0: {Simplex((1, 2)), Simplex((1,)), Simplex((2,))}})

defaultdict(set,
            {0.0: {Simplex((1, 2)), Simplex((1,)), Simplex((2,))},
             1.0: {Simplex((1, 2, 3)),
              Simplex((1, 3)),
              Simplex((2, 3)),
              Simplex((3,))}})

ValueError("'generator' contains a face with a bad time value")

**Ejercicio 11:** Añade un nuevo atributo a la clase que almacene la lista de todos los símplices ordenados según el valor del flotante correspondiente teniendo ordenando dos que tengan el mismo flotante asociado por su dimensión.

Este ejercicio se resuelve usando una función `filtration_order` anotada con `@property`, en vez de con un atributo.

La expresión `timestamps.items()` devuelve un iterable de elementos de la forma `(time, {simplices})`. Así, ordenando este iterable usando `sorted` obtendremos otro iterable de la misma forma pero ordenado según `time` de menor a mayor. Por último, como cada `v` es un iterable de símplices y cada uno de ellos lleva asociado el mismo flotante, se ordenan por su dimensión y desempatando con el orden lexicográfico de sus vértices.

In [29]:
    @property
    def filtration_order(self) -> list[Simplex]:
        """Return the filtration order as a sorted list of simplices."""
        return [
            simplex 
            for _, v in sorted(self.timestamps.items()) 
            for simplex in sorted(v, key=lambda s: (s.dim(), s.vertices))
        ]

De este modo, la clase queda así:

In [30]:
class FilteredSimplicialComplex:
    """Represents a simplicial complex whose simplices carry filtration timestamps."""

    def _finish_insertion(self, additions: set[Simplex], time: float) -> None:
        """Add simplices to the timestamp dictionary."""
        if not additions:
            return
            
        self.timestamps[time].update(additions)

        for addition in additions:
            self.time_of[addition] = time

    def __init__(self, generator: Iterable[Simplex], time: float = 0.0) -> None:
        """Initialize the filtered complex with a set of simplices."""
        self.sc: SimplicialComplex = SimplicialComplex(generator)

        self.timestamps: dict[float, set[Simplex]] = defaultdict(set)
        self.time_of: dict[Simplex, float] = {}

        self._finish_insertion(self.sc.face_set, time)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the face set."""
        return self.sc.face_set

    @property
    def filtration_order(self) -> list[Simplex]:
        """Return the filtration order as a sorted list of simplices."""
        return [
            simplex 
            for _, v in sorted(self.timestamps.items()) 
            for simplex in sorted(v, key=lambda s: (s.dim(), s.vertices))
        ]
    
    def insert(self, generator: Iterable[Simplex], time: float) -> None:
        """Insert a generator of simplices at the given timestamp."""
        generator = list(generator)

        for simplex in generator:
            if simplex in self.sc.face_set:
                raise ValueError("'generator' contains an already added simplex")

        additions: set[Simplex] = set()

        for simplex in generator:
            additions.update(simplex.faces())

        additions -= self.sc.face_set

        for addition in additions:
            for face in addition.faces():
                t = self.time_of.get(face)

                if t is not None and t > time:
                    raise ValueError("'generator' contains a face with a bad time value")

        self.sc.insert(generator)
        self._finish_insertion(additions, time)

Se comprueba su funcionamiento:

In [31]:
fsc = FilteredSimplicialComplex([])

fsc.insert([Simplex((1,))], 0.11)
fsc.insert([Simplex((1,2))], 0.22)
fsc.insert([Simplex((1,2,3))], 0.33)

fsc.filtration_order

[Simplex((1,)),
 Simplex((2,)),
 Simplex((1, 2)),
 Simplex((3,)),
 Simplex((1, 3)),
 Simplex((2, 3)),
 Simplex((1, 2, 3))]

**Ejercicio 12:** Crea una función que recupere el complejo simplicial formado por todos los 
símplices cuyo flotante asociado sea menor o igual que un flotante dado.

Primero se itera sobre `timestamps.items`, que entrega valores de la forma `(time, {simplices})`. Después, se filtran aquellos con un flotante sea menor o igual al dado y se itera sobre los símplices del conjunto `simplices`. Todo esto genera un iterable de símplices con flotante asociado menor o igual a `time`. Por último, se crea el complejo simplicial formado por estos símplices usando la clase `SimplicialComplex`.

In [32]:
    def filtration(self, time: float) -> SimplicialComplex:
        """Return the simplicial complex at the given timestamp."""
        return SimplicialComplex(
            simplex 
            for k, v in self.timestamps.items() 
            if k <= time 
            for simplex in v
        )

Adicionalmente, se implementan los métodos complementarios `threshold`, que retorna el instante en el que aparece el símplice provisto, y `threshold_values`, que retorna todos los instantes en los que aparece un nuevo símplice ordenados de menor a mayor.

In [33]:
    def threshold(self, simplex: Simplex) -> Optional[float]:
        """Compute the threshold value of a simplex."""
        return self.time_of.get(simplex)

    def threshold_values(self) -> list[float]:
        """Return the sorted list of all threshold values."""
        return sorted(self.timestamps.keys())

De este modo, la clase `FilteredSimplicialComplex` termina siendo así:

In [34]:
class FilteredSimplicialComplex:
    """Represents a simplicial complex whose simplices carry filtration timestamps."""

    def _finish_insertion(self, additions: set[Simplex], time: float) -> None:
        """Add simplices to the timestamp dictionary."""
        if not additions:
            return
            
        self.timestamps[time].update(additions)

        for addition in additions:
            self.time_of[addition] = time

    def __init__(self, generator: Iterable[Simplex], time: float = 0.0) -> None:
        """Initialize the filtered complex with a set of simplices."""
        self.sc: SimplicialComplex = SimplicialComplex(generator)

        self.timestamps: dict[float, set[Simplex]] = defaultdict(set)
        self.time_of: dict[Simplex, float] = {}

        self._finish_insertion(self.sc.face_set, time)

    @property
    def face_set(self) -> set[Simplex]:
        """Return the face set."""
        return self.sc.face_set

    @property
    def filtration_order(self) -> list[Simplex]:
        """Return the filtration order as a sorted list of simplices."""
        return [
            simplex 
            for _, v in sorted(self.timestamps.items()) 
            for simplex in sorted(v, key=lambda s: (s.dim(), s.vertices))
        ]
    
    def insert(self, generator: Iterable[Simplex], time: float) -> None:
        """Insert a generator of simplices at the given timestamp."""
        generator = list(generator)

        for simplex in generator:
            if simplex in self.sc.face_set:
                raise ValueError("'generator' contains an already added simplex")

        additions: set[Simplex] = set()

        for simplex in generator:
            additions.update(simplex.faces())

        additions -= self.sc.face_set

        for addition in additions:
            for face in addition.faces():
                t = self.time_of.get(face)

                if t is not None and t > time:
                    raise ValueError("'generator' contains a face with a bad time value")

        self.sc.insert(generator)
        self._finish_insertion(additions, time)

    def filtration(self, time: float) -> SimplicialComplex:
        """Return the simplicial complex at the given timestamp."""
        return SimplicialComplex(
            simplex 
            for k, v in self.timestamps.items() 
            if k <= time 
            for simplex in v
        )

    def threshold(self, simplex: Simplex) -> Optional[float]:
        """Compute the threshold value of a simplex."""
        return self.time_of.get(simplex)

    def threshold_values(self) -> list[float]:
        """Return the sorted list of all threshold values."""
        return sorted(self.timestamps.keys())

Se verifica el funcionamiento de los nuevos métodos:

In [35]:
fsc = FilteredSimplicialComplex([])

fsc.insert([Simplex((1,))], 0.11)
fsc.insert([Simplex((1,2))], 0.22)
fsc.insert([Simplex((1,2,3))], 0.33)

display(fsc.filtration(0.11).face_set)
display(fsc.filtration(0.5).face_set)

{Simplex((1,))}

{Simplex((1, 2)),
 Simplex((1, 2, 3)),
 Simplex((1, 3)),
 Simplex((1,)),
 Simplex((2, 3)),
 Simplex((2,)),
 Simplex((3,))}

In [36]:
fsc.threshold(Simplex((2,)))

0.22

In [37]:
fsc.threshold_values()

[0.11, 0.22, 0.33]